# 01b — Machine Calibration (Linear Test)

This section processes the **linear test** dataset into machine calibration values.

**Flow:**
1. Load `K` and `D` from `01_camera_calibration/camera_calibration.ipynb`
2. Read image triplets `oriy{i}.png` / `fwd{i}.png` / `bck{i}.png` (i = 1..40)
3. `solvePnP` per image -> `tvec` (position in mm), filtered by an RMSE threshold
4. Compute mean dX/dY, error against the target, backlash, and skew
5. Print firmware recommendations: `M92` (steps/mm), `M425` (backlash), `M852` (skew)

**How to use baseline vs calibration results:** first run with `CURRENT_STEPS_MM`
and the baseline folder, then switch `base_path` + `CURRENT_STEPS_MM` to the calibration-result
folder to see the residual error after calibration.

> Cells 10 (`FIND NOISE`) and 11 (`cek_frame_folder`) are helper utilities,
> not part of the calibration computation.


In [ ]:
# ==========================================================================
# 1. LOAD PATH CONFIGURATION  (single source: config_paths.py at the repo root)
# ==========================================================================
import os
import sys

_p = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(_p, "config_paths.py")):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError(
            "config_paths.py not found. Run this notebook from inside the "
            "organized repo folder, or set the KALIBRASI_REPO environment variable to the repo root."
        )
    _p = _parent
sys.path.insert(0, _p)
import config_paths as CFG

from calibration_pipeline import (
    detect_edlines_raw, get_initial_corners_strict_paper,
    optimize_subpixel_gradient, sort_corners_robust_paper,
)

CFG.diagnose()
print()

# ==========================================================================
# 2. MACHINE PARAMETERS PER TEST MODE
# ==========================================================================
# Switch modes without editing the notebook:
#     Windows : set KALIBRASI_MODE=baseline
#     Linux   : export KALIBRASI_MODE=baseline
#   baseline          -> before calibration, steps/mm 80,          target 10 mm
#   calibrated        -> after calibration, steps/mm 79.72/80.40,  target 10 mm
#   measurement_error -> 0.5 mm dataset, all baseline, dial gauge reference
RUN_MODE = CFG.RUN_MODE
P = CFG.MODE_PARAMS[RUN_MODE]
print(f"MODE: {RUN_MODE}  ->  delta_x={P['delta_x']}  delta_y={P['delta_y']}  "
      f"steps_x={P['steps_x']}  steps_y={P['steps_y']}")
print(f"  dataset X : {CFG.DATASET_X}")
print(f"  dataset Y : {CFG.DATASET_Y}")
print()

# ==========================================================================
# 3. LOAD CAMERA PARAMETERS (K and D) FROM THE CAMERA CALIBRATION CSV
# ==========================================================================
# Read from the CSV output of 01_camera_calibration/camera_calibration.ipynb,
# so the camera calibration does not need to be re-run.
import numpy as np
import pandas as pd

CAM_DIR = os.path.join(CFG.OUTPUT_DIR, "camera_calibration")
_HINT = ("Run 01_camera_calibration/camera_calibration.ipynb first until "
         "the export cell finishes so the camera parameter CSVs are created.")

if "camera_matrix" in globals() and "dist_coeffs" in globals():
    print("Camera parameters already exist in kernel memory.")
else:
    for _f in ("camera_matrix.csv", "dist_coeffs.csv", "camera_params.csv"):
        if not os.path.isfile(os.path.join(CAM_DIR, _f)):
            raise FileNotFoundError(f"Not found: {os.path.join(CAM_DIR, _f)}\n{_HINT}")

    camera_matrix = np.loadtxt(os.path.join(CAM_DIR, "camera_matrix.csv"),
                               delimiter=",", ndmin=2)
    dist_coeffs = np.loadtxt(os.path.join(CAM_DIR, "dist_coeffs.csv"),
                             delimiter=",", ndmin=2).reshape(1, -1)

    _p = pd.read_csv(os.path.join(CAM_DIR, "camera_params.csv")).set_index("parameter")["value"]
    img_shape = (int(_p["image_width"]), int(_p["image_height"]))

    print(f"Camera parameters loaded from: {CAM_DIR}")
    print(f"  image_size  : {img_shape}")
    print("  K =\n", np.round(camera_matrix, 4))
    print("  dist_coeffs =", np.round(dist_coeffs.ravel(), 6))
print()


In [ ]:
# POSITION EXTRACTION, STATISTICAL ANALYSIS, AND FIRMWARE CALIBRATION (STEPS/MM, BACKLASH, SKEW) FOR X
# WITH AUTOMATIC VISUALIZATION ON THE FIRST VALID SAMPLE
# =========================================================================================
import cv2
import numpy as np
import os

# --- 1. LOADING CAMERA PARAMETERS ---
try:
    K = camera_matrix 
    D = dist_coeffs
except NameError:
    raise NameError("ERROR: Run the Camera Calibration Cell first!")

CHESSBOARD_SIZE = CFG.CHESSBOARD_SIZE
SQUARE_SIZE = CFG.SQUARE_SIZE_MM  # mm
expected_pts = CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1]

objp = np.zeros((expected_pts, 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2) * SQUARE_SIZE

# --- 2. INPUT CONFIGURATION & MACHINE PARAMETERS ---
base_path = CFG.DATASET_X   # set via config_paths.py

# Target & Machine Parameters
EXPECTED_DELTA_X = P["delta_x"] if True else 0.0
EXPECTED_DELTA_Y = P["delta_y"] if False else 0.0
CURRENT_STEPS_MM = P["steps_x"]
RMSE_THRESHOLD = 0.2       # Maximum allowed RMSE
MAX_VALID_SAMPLES = 10     # Required number of valid samples

FILE_INDICES = list(range(1, 41))

# Build image triplets: (Origin, Forward, Backward/Return)
IMAGE_SETS = [
    (
        os.path.join(base_path, f"oriy{i}.png"),
        os.path.join(base_path, f"fwd{i}.png"),
        os.path.join(base_path, f"bck{i}.png")
    )
    for i in FILE_INDICES
]

# --- 3. POSE EXTRACTION FUNCTION (WITH INTEGRATED DRAWING FEATURE) ---
def get_pose_from_image(image_path, draw_debug=False):
    if not os.path.exists(image_path):
        return None, None, None
        
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Create the drawing canvas: grayscale but 3-channel so red can be drawn
    img_draw = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR) if draw_debug else None
    
    # EDLines pipeline (assumes these functions already exist in memory/previous cell)
    lines = detect_edlines_raw(gray)
    initial_corners, _ = get_initial_corners_strict_paper(gray, lines)
    
    if len(initial_corners) >= expected_pts:
        refined_corners = optimize_subpixel_gradient(gray, initial_corners)
        sorted_corners = sort_corners_robust_paper(refined_corners, gray, CHESSBOARD_SIZE)
        
        if sorted_corners is not None:
            # --- CORNER LAYOUT DRAWING BLOCK ---
            if draw_debug:
                for idx, pt in enumerate(sorted_corners):
                    # Draw corner points (red)
                    cv2.circle(img_draw, (int(pt[0]), int(pt[1])), 2, (0, 0, 255), -1)
                    # Add corner order index text
                    cv2.putText(img_draw, str(idx+1), (int(pt[0])+4, int(pt[1])-4), 1, 1.5, (0, 0, 255), 2)

            corners_final = sorted_corners.reshape(-1, 1, 2)
            success, rvec, tvec = cv2.solvePnP(objp, corners_final, K, D)
            
            if success:
                imgpts2, _ = cv2.projectPoints(objp, rvec, tvec, K, D)
                norm_l2 = cv2.norm(corners_final, imgpts2, cv2.NORM_L2)
                rmse = np.sqrt((norm_l2 ** 2) / len(imgpts2))
                return tvec, rmse, img_draw
                
    return None, None, img_draw

# --- 4. BATCH EXECUTION & ANALYSIS ---
print("="*70)
print("POSITION, TRAVEL & REPROJECTION ERROR ANALYSIS")
print("="*70)

# Statistic data accumulator lists
list_delta_x = []
list_delta_y = []
list_error_x = []
list_error_y = []
list_rmse = []
list_backlash = []
list_skew = []

# Failed data accumulator list
list_sampel_gagal = []
berhasil = 0

# Flag indicating whether the first valid sample visualization has been saved
visualisasi_tersimpan = False

for idx, (ori_path, fwd_path, bck_path) in enumerate(IMAGE_SETS):
    sampel_id = FILE_INDICES[idx]
    print(f"\n[SAMPLE {sampel_id}]")
    
    # Enable visualization mode if no valid sample has been saved yet
    is_visualize = not visualisasi_tersimpan
    
    tvec_ori, rmse_ori, img_ori_drawn = get_pose_from_image(ori_path, draw_debug=is_visualize)
    tvec_fwd, rmse_fwd, img_fwd_drawn = get_pose_from_image(fwd_path, draw_debug=is_visualize)
    tvec_bck, rmse_bck, img_bck_drawn = get_pose_from_image(bck_path, draw_debug=is_visualize)
    
    # --- VALIDATION LOGIC ---
    # 1. Check whether any image failed to process (None)
    if tvec_ori is None or tvec_fwd is None or tvec_bck is None:
        print("  [FAILED] One or more images could not be processed (no pose found).")
        list_sampel_gagal.append(sampel_id)
        continue 
        
    # 2. Check whether any RMSE exceeds the threshold
    if rmse_ori > RMSE_THRESHOLD or rmse_fwd > RMSE_THRESHOLD or rmse_bck > RMSE_THRESHOLD:
        print(f"  [FAILED] Rejected: RMSE > {RMSE_THRESHOLD}.")
        print(f"          RMSE detail -> Ori: {rmse_ori:.4f}, Fwd: {rmse_fwd:.4f}, Bck: {rmse_bck:.4f}")
        list_sampel_gagal.append(sampel_id)
        continue 

    # --- IF VALIDATION PASSES, PROCESS THE DATA ---
    berhasil += 1
    
    x_orig, y_orig = tvec_ori[0][0], tvec_ori[1][0]
    x_fwd, y_fwd = tvec_fwd[0][0], tvec_fwd[1][0]
    x_bck, y_bck = tvec_bck[0][0], tvec_bck[1][0]
    
    # Forward Delta computation
    delta_x = x_fwd - x_orig
    delta_y = y_fwd - y_orig
    
    # Error against Expected computation
    error_x = abs(delta_x - EXPECTED_DELTA_X)
    error_y = abs(delta_y - EXPECTED_DELTA_Y)
    
    # Backlash computation
    backlash_val = abs(x_bck - x_orig)
    
    # Skew Factor computation (Y drift / X move)
    skew_factor = delta_y / delta_x if delta_x != 0 else 0
    
    # Store basic metrics into the lists
    list_delta_x.append(delta_x)
    list_delta_y.append(delta_y)
    list_error_x.append(error_x)
    list_error_y.append(error_y)
    list_skew.append(skew_factor)
    list_backlash.append(backlash_val)
    list_rmse.extend([rmse_ori, rmse_fwd, rmse_bck])
    
    # Screen output
    print(f"  > Initial position  : X = {x_orig:8.4f} mm, Y = {y_orig:8.4f} mm | RMSE = {rmse_ori:.4f} px")
    print(f"  > Forward position  : X = {x_fwd:8.4f} mm, Y = {y_fwd:8.4f} mm | RMSE = {rmse_fwd:.4f} px")
    print(f"  > Backward position : X = {x_bck:8.4f} mm, Y = {y_bck:8.4f} mm | RMSE = {rmse_bck:.4f} px")
    print(f"  > TRAVEL        : dX = {delta_x:8.4f} mm (Err: {error_x:.4f}), dY = {delta_y:8.4f} mm (Err: {error_y:.4f})")
    print(f"  > BACKLASH (X)  : {backlash_val:8.4f} mm")

    # # --- VISUALIZATION & PNG SAVE BLOCK (FIRST VALID SAMPLE ONLY) THIS FONT IS TOO SMALL ---
    # if is_visualize and img_ori_drawn is not None and img_fwd_drawn is not None and img_bck_drawn is not None:
    #     font = cv2.FONT_HERSHEY_SIMPLEX
        
    #     # Text Overlay - Origin Frame
    #     cv2.putText(img_ori_drawn, "Origin Frame", (30, 50), font, 1.2, (0, 0, 255), 3)
    #     cv2.putText(img_ori_drawn, f"X: {x_orig:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
    #     cv2.putText(img_ori_drawn, f"Y: {y_orig:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
    #     cv2.putText(img_ori_drawn, f"RMSE: {rmse_ori:.4f} px", (30, 170), font, 0.9, (255, 255, 0), 2)
        
    #     # Text Overlay - Forward Frame
    #     cv2.putText(img_fwd_drawn, "Forward Frame", (30, 50), font, 1.2, (0, 0, 255), 3)
    #     cv2.putText(img_fwd_drawn, f"X: {x_fwd:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
    #     cv2.putText(img_fwd_drawn, f"Y: {y_fwd:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
    #     cv2.putText(img_fwd_drawn, f"RMSE: {rmse_fwd:.4f} px", (30, 170), font, 0.9, (255, 255, 0), 2)
        
    #     # Text Overlay - Backward Frame
    #     cv2.putText(img_bck_drawn, "Backward Frame", (30, 50), font, 1.2, (0, 0, 255), 3)
    #     cv2.putText(img_bck_drawn, f"X: {x_bck:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
    #     cv2.putText(img_bck_drawn, f"Y: {y_bck:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
    #     cv2.putText(img_bck_drawn, f"RMSE: {rmse_bck:.4f} px", (30, 170), font, 0.9, (255, 255, 0), 2)
        
    #     # Save the three image files
    #     cv2.imwrite("visualization_origin.png", img_ori_drawn)
    #     cv2.imwrite("visualization_forward.png", img_fwd_drawn)
    #     cv2.imwrite("visualization_backward.png", img_bck_drawn)
        
    #     print("\n  [INFO] Images 'visualization_origin.png', 'visualization_forward.png', and 'visualization_backward.png' saved successfully!")

    # --- VISUALIZATION & PNG SAVE BLOCK (FIRST VALID SAMPLE ONLY) ---
    if is_visualize and img_ori_drawn is not None and img_fwd_drawn is not None and img_bck_drawn is not None:
        font = cv2.FONT_HERSHEY_SIMPLEX
        
        # Font Scale and Thickness Settings (Enlarged)
        scale_title = 1.8   # The larger the value, the larger the title
        scale_text = 1.3    # The larger the value, the larger the data text
        thick_title = 4
        thick_text = 3

        # Text Overlay - Origin Frame
        cv2.putText(img_ori_drawn, "Origin Frame", (30, 60), font, scale_title, (0, 0, 255), thick_title)
        cv2.putText(img_ori_drawn, f"X: {x_orig:.3f} mm", (30, 120), font, scale_text, (0, 255, 0), thick_text)
        cv2.putText(img_ori_drawn, f"Y: {y_orig:.3f} mm", (30, 180), font, scale_text, (0, 255, 0), thick_text)
        cv2.putText(img_ori_drawn, f"RMSE: {rmse_ori:.4f} px", (30, 240), font, scale_text, (255, 255, 0), thick_text)
        
        # Text Overlay - Forward Frame
        cv2.putText(img_fwd_drawn, "Forward Frame", (30, 60), font, scale_title, (0, 0, 255), thick_title)
        cv2.putText(img_fwd_drawn, f"X: {x_fwd:.3f} mm", (30, 120), font, scale_text, (0, 255, 0), thick_text)
        cv2.putText(img_fwd_drawn, f"Y: {y_fwd:.3f} mm", (30, 180), font, scale_text, (0, 255, 0), thick_text)
        cv2.putText(img_fwd_drawn, f"RMSE: {rmse_fwd:.4f} px", (30, 240), font, scale_text, (255, 255, 0), thick_text)
        
        # Text Overlay - Backward Frame
        cv2.putText(img_bck_drawn, "Backward Frame", (30, 60), font, scale_title, (0, 0, 255), thick_title)
        cv2.putText(img_bck_drawn, f"X: {x_bck:.3f} mm", (30, 120), font, scale_text, (0, 255, 0), thick_text)
        cv2.putText(img_bck_drawn, f"Y: {y_bck:.3f} mm", (30, 180), font, scale_text, (0, 255, 0), thick_text)
        cv2.putText(img_bck_drawn, f"RMSE: {rmse_bck:.4f} px", (30, 240), font, scale_text, (255, 255, 0), thick_text)
        
        # Save the three image files
        cv2.imwrite("visualization_origin.png", img_ori_drawn)
        cv2.imwrite("visualization_forward.png", img_fwd_drawn)
        cv2.imwrite("visualization_backward.png", img_bck_drawn)
        
        print("\n  [INFO] Images 'visualization_origin.png', 'visualization_forward.png', and 'visualization_backward.png' saved successfully!")
        
        # Mark that the visualization has been captured to prevent draw_debug in later iterations
        visualisasi_tersimpan = True

    # --- CHECK MAXIMUM SAMPLE LIMIT ---
    if berhasil >= MAX_VALID_SAMPLES:
        print(f"\n[INFO] Reached the target of {MAX_VALID_SAMPLES} valid samples. Extraction stopped.")
        break

# --- 5. STATISTICAL SUMMARY & FIRMWARE RECOMMENDATIONS ---
print("\n" + "="*70)
print("FINAL STATISTICS & FIRMWARE CALIBRATION RECOMMENDATIONS")
print("="*70)

if berhasil > 0:
    # Mean & Standard Deviation Statistics Computation
    avg_dx, std_dx = np.mean(list_delta_x), np.std(list_delta_x)
    avg_dy, std_dy = np.mean(list_delta_y), np.std(list_delta_y)
    
    avg_err_x, std_err_x = np.mean(list_error_x), np.std(list_error_x)
    avg_err_y, std_err_y = np.mean(list_error_y), np.std(list_error_y)
    
    avg_rmse, std_rmse = np.mean(list_rmse), np.std(list_rmse)
    
    print(f"Valid samples: {berhasil} of {len(IMAGE_SETS)} available\n")
    
    print("--- 1. TRAVEL & ERROR STATISTICS ---")
    print(f"Target               : X = {EXPECTED_DELTA_X:.4f} mm, Y = {EXPECTED_DELTA_Y:.4f} mm")
    print(f"Mean Delta X         : {avg_dx:8.4f} mm (Std Dev: {std_dx:.4f} mm)")
    print(f"Mean Delta Y         : {avg_dy:8.4f} mm (Std Dev: {std_dy:.4f} mm)")
    print(f"Mean Error X         : {avg_err_x:8.4f} mm (Std Dev: {std_err_x:.4f} mm)")
    print(f"Mean Error Y         : {avg_err_y:8.4f} mm (Std Dev: {std_err_y:.4f} mm)")
    print(f"Mean RMSE            : {avg_rmse:8.4f} px (Std Dev: {std_rmse:.4f} px)")
    
    print("\n--- 2. STEPS/MM UPDATE RECOMMENDATION ---")
    if EXPECTED_DELTA_X != 0:
        new_steps_x = (EXPECTED_DELTA_X / avg_dx) * CURRENT_STEPS_MM
        print(f"Current Steps/mm (X) : {CURRENT_STEPS_MM:.4f}")
        print(f"-> NEW Steps/mm (X)  : {new_steps_x:.4f}  (G-Code: M92 X{new_steps_x:.2f})")
    else:
        print("-> Steps/mm X update: skipped (target Delta X = 0)")

    print("\n--- 3. BACKLASH COMPENSATION RECOMMENDATION ---")
    avg_backlash = np.mean(list_backlash)
    std_backlash = np.std(list_backlash)
    print(f"Mean Backlash X      : {avg_backlash:8.4f} mm (Std Dev: {std_backlash:.4f} mm)")
    print(f"-> Firmware update : G-Code: M425 X{avg_backlash:.4f}")

    print("\n--- 4. SKEW COMPENSATION RECOMMENDATION (XY ORTHOGONALITY) ---")
    avg_skew = np.mean(list_skew)
    angle_deg = np.degrees(np.arctan(avg_skew))
    print(f"Mean Skew (I)        : {avg_skew:.6f}")
    print(f"Axis tilt            : {angle_deg:.4f} Degrees")
    if abs(avg_skew) < 0.001:
        print("-> Skew status      : Orthogonal. No compensation needed.")
    else:
        print(f"-> Firmware update : G-Code: M852 I{avg_skew:.6f}")

else:
    print("No valid data passed the selection (RMSE > 0.2 or processing failed).")

# --- DISPLAY FAILED SAMPLES ---
print("\n--- 5. FAILED SAMPLE STATUS ---")
if len(list_sampel_gagal) > 0:
    print(f"There were {len(list_sampel_gagal)} rejected/failed samples before reaching {MAX_VALID_SAMPLES} valid samples:")
    print(f"-> {list_sampel_gagal}")
else:
    print(f"-> No samples failed before reaching the target of {MAX_VALID_SAMPLES} valid samples!")
print("="*70)

In [ ]:
# POSITION EXTRACTION, STATISTICAL ANALYSIS, & Y-AXIS CALIBRATION
# WITH AUTOMATIC VISUALIZATION ON THE FIRST VALID SAMPLE
# =========================================================
import cv2
import numpy as np
import os

# --- 1. LOADING CAMERA PARAMETERS ---
try:
    K = camera_matrix 
    D = dist_coeffs
except NameError:
    raise NameError("ERROR: Run the Camera Calibration Cell first!")

CHESSBOARD_SIZE = CFG.CHESSBOARD_SIZE
SQUARE_SIZE = CFG.SQUARE_SIZE_MM  # mm
expected_pts = CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1]

objp = np.zeros((expected_pts, 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2) * SQUARE_SIZE

# --- 2. INPUT CONFIGURATION & MACHINE PARAMETERS ---
# Adjust base_path to the Y-Axis dataset folder
base_path = CFG.DATASET_Y   # set via config_paths.py

# Target & Machine Parameters (Y-Axis Focus)
EXPECTED_DELTA_X = P["delta_x"] if False else 0.0
EXPECTED_DELTA_Y = P["delta_y"] if True else 0.0
CURRENT_STEPS_MM = P["steps_y"]
RMSE_THRESHOLD = 0.2       # Maximum allowed RMSE
MAX_VALID_SAMPLES = 10     # Required number of valid samples

FILE_INDICES = list(range(1, 41))

# Build image triplets for the Y axis: (Origin, Forward, Backward/Return)
IMAGE_SETS = [
    (
        os.path.join(base_path, f"oriy{i}.png"),
        os.path.join(base_path, f"fwd{i}.png"),
        os.path.join(base_path, f"bck{i}.png")
    )
    for i in FILE_INDICES
]

# --- 3. POSE EXTRACTION FUNCTION (WITH INTEGRATED DRAWING FEATURE) ---
def get_pose_from_image(image_path, draw_debug=False):
    if not os.path.exists(image_path):
        return None, None, None
        
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Create the drawing canvas: grayscale but 3-channel so red can be drawn
    img_draw = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR) if draw_debug else None
    
    # EDLines pipeline (assumes these functions already exist in memory/previous cell)
    lines = detect_edlines_raw(gray)
    initial_corners, _ = get_initial_corners_strict_paper(gray, lines)
    
    if len(initial_corners) >= expected_pts:
        refined_corners = optimize_subpixel_gradient(gray, initial_corners)
        sorted_corners = sort_corners_robust_paper(refined_corners, gray, CHESSBOARD_SIZE)
        
        if sorted_corners is not None:
            # --- CORNER LAYOUT DRAWING BLOCK ---
            if draw_debug:
                for idx, pt in enumerate(sorted_corners):
                    # Draw corner points (red)
                    cv2.circle(img_draw, (int(pt[0]), int(pt[1])), 2, (0, 0, 255), -1)
                    # Add corner order index text
                    cv2.putText(img_draw, str(idx+1), (int(pt[0])+4, int(pt[1])-4), 1, 1.5, (0, 0, 255), 2)

            corners_final = sorted_corners.reshape(-1, 1, 2)
            success, rvec, tvec = cv2.solvePnP(objp, corners_final, K, D)
            
            if success:
                imgpts2, _ = cv2.projectPoints(objp, rvec, tvec, K, D)
                norm_l2 = cv2.norm(corners_final, imgpts2, cv2.NORM_L2)
                rmse = np.sqrt((norm_l2 ** 2) / len(imgpts2))
                return tvec, rmse, img_draw
                
    return None, None, img_draw

# --- 4. BATCH EXECUTION & ANALYSIS ---
print("="*70)
print("POSITION, TRAVEL & REPROJECTION ERROR ANALYSIS (Y AXIS)")
print("="*70)

# Statistic data accumulator lists
list_delta_x = []
list_delta_y = []
list_error_x = []
list_error_y = []
list_rmse = []
list_backlash = []
list_skew = []

# Failed data accumulator list
list_sampel_gagal = []
berhasil = 0

# Flag indicating whether the first valid sample visualization has been saved
visualisasi_tersimpan = False

for idx, (ori_path, fwd_path, bck_path) in enumerate(IMAGE_SETS):
    sampel_id = FILE_INDICES[idx]
    print(f"\n[SAMPLE {sampel_id}]")
    
    # Enable visualization mode if no valid sample has been saved yet
    is_visualize = not visualisasi_tersimpan
    
    tvec_ori, rmse_ori, img_ori_drawn = get_pose_from_image(ori_path, draw_debug=is_visualize)
    tvec_fwd, rmse_fwd, img_fwd_drawn = get_pose_from_image(fwd_path, draw_debug=is_visualize)
    tvec_bck, rmse_bck, img_bck_drawn = get_pose_from_image(bck_path, draw_debug=is_visualize)
    
    # --- VALIDATION LOGIC ---
    # 1. Check whether any image failed to process (None)
    if tvec_ori is None or tvec_fwd is None or tvec_bck is None:
        print("  [FAILED] One or more images could not be processed (no pose found).")
        list_sampel_gagal.append(sampel_id)
        continue  # Immediately proceed to the next sample iteration
        
    # 2. Check whether any RMSE exceeds the threshold
    if rmse_ori > RMSE_THRESHOLD or rmse_fwd > RMSE_THRESHOLD or rmse_bck > RMSE_THRESHOLD:
        print(f"  [FAILED] Rejected: RMSE > {RMSE_THRESHOLD}.")
        print(f"          RMSE detail -> Ori: {rmse_ori:.4f}, Fwd: {rmse_fwd:.4f}, Bck: {rmse_bck:.4f}")
        list_sampel_gagal.append(sampel_id)
        continue  # Immediately proceed to the next sample iteration

    # --- IF VALIDATION PASSES, PROCESS THE DATA ---
    berhasil += 1
    
    x_orig, y_orig = tvec_ori[0][0], tvec_ori[1][0]
    x_fwd, y_fwd = tvec_fwd[0][0], tvec_fwd[1][0]
    x_bck, y_bck = tvec_bck[0][0], tvec_bck[1][0]
    
    # Forward Delta computation
    delta_x = x_fwd - x_orig
    delta_y = y_fwd - y_orig
    
    # Error against Expected computation
    error_x = abs(delta_x - EXPECTED_DELTA_X)
    error_y = abs(delta_y - EXPECTED_DELTA_Y)
    
    # Backlash computation (Y-Axis focus: difference between final Y and initial Y)
    backlash_val = abs(y_bck - y_orig)
    
    # Kalkulasi Skew Factor (X drift / Y move)
    # When moving along the Y axis, skew is how far the X axis drifts
    skew_factor = delta_x / delta_y if delta_y != 0 else 0
    
    # Store basic metrics into the lists
    list_delta_x.append(delta_x)
    list_delta_y.append(delta_y)
    list_error_x.append(error_x)
    list_error_y.append(error_y)
    list_skew.append(skew_factor)
    list_backlash.append(backlash_val)
    list_rmse.extend([rmse_ori, rmse_fwd, rmse_bck])
    
    # Screen output
    print(f"  > Initial position  : X = {x_orig:8.4f} mm, Y = {y_orig:8.4f} mm | RMSE = {rmse_ori:.4f} px")
    print(f"  > Forward position  : X = {x_fwd:8.4f} mm, Y = {y_fwd:8.4f} mm | RMSE = {rmse_fwd:.4f} px")
    print(f"  > Backward position : X = {x_bck:8.4f} mm, Y = {y_bck:8.4f} mm | RMSE = {rmse_bck:.4f} px")
    print(f"  > TRAVEL        : dX = {delta_x:8.4f} mm (Err: {error_x:.4f}), dY = {delta_y:8.4f} mm (Err: {error_y:.4f})")
    print(f"  > BACKLASH (Y)  : {backlash_val:8.4f} mm")

    # --- VISUALIZATION & PNG SAVE BLOCK (FIRST VALID SAMPLE ONLY) ---
    if is_visualize and img_ori_drawn is not None and img_fwd_drawn is not None and img_bck_drawn is not None:
        font = cv2.FONT_HERSHEY_SIMPLEX
        
        # Text Overlay - Origin Frame
        cv2.putText(img_ori_drawn, "ORIGIN FRAME", (30, 50), font, 1.2, (0, 0, 255), 3)
        cv2.putText(img_ori_drawn, f"X: {x_orig:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
        cv2.putText(img_ori_drawn, f"Y: {y_orig:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
        cv2.putText(img_ori_drawn, f"RMSE: {rmse_ori:.4f} px", (30, 170), font, 0.9, (255, 255, 0), 2)
        
        # Text Overlay - Forward Frame
        cv2.putText(img_fwd_drawn, "FORWARD FRAME", (30, 50), font, 1.2, (0, 0, 255), 3)
        cv2.putText(img_fwd_drawn, f"X: {x_fwd:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
        cv2.putText(img_fwd_drawn, f"Y: {y_fwd:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
        cv2.putText(img_fwd_drawn, f"RMSE: {rmse_fwd:.4f} px", (30, 170), font, 0.9, (255, 255, 0), 2)
        
        # Text Overlay - Backward Frame
        cv2.putText(img_bck_drawn, "BACKWARD FRAME", (30, 50), font, 1.2, (0, 0, 255), 3)
        cv2.putText(img_bck_drawn, f"X: {x_bck:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
        cv2.putText(img_bck_drawn, f"Y: {y_bck:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
        cv2.putText(img_bck_drawn, f"RMSE: {rmse_bck:.4f} px", (30, 170), font, 0.9, (255, 255, 0), 2)
        
        # Save the three image files (tagged _Y_ so they do not overwrite the X axis)
        cv2.imwrite("visualization_Y_origin.png", img_ori_drawn)
        cv2.imwrite("visualization_Y_forward.png", img_fwd_drawn)
        cv2.imwrite("visualization_Y_backward.png", img_bck_drawn)
        
        print("\n  [INFO] Images 'visualization_Y_origin.png', 'visualization_Y_forward.png', and 'visualization_Y_backward.png' saved successfully!")
        
        # Mark that the visualization has been captured to prevent draw_debug in later iterations
        visualisasi_tersimpan = True

    # --- CHECK MAXIMUM SAMPLE LIMIT ---
    if berhasil >= MAX_VALID_SAMPLES:
        print(f"\n[INFO] Reached the target of {MAX_VALID_SAMPLES} valid samples. Extraction stopped.")
        break

# --- 5. STATISTICAL SUMMARY & FIRMWARE RECOMMENDATIONS ---
print("\n" + "="*70)
print("FINAL STATISTICS & FIRMWARE CALIBRATION RECOMMENDATIONS")
print("="*70)

if berhasil > 0:
    # Mean & Standard Deviation Statistics Computation
    avg_dx, std_dx = np.mean(list_delta_x), np.std(list_delta_x)
    avg_dy, std_dy = np.mean(list_delta_y), np.std(list_delta_y)
    
    avg_err_x, std_err_x = np.mean(list_error_x), np.std(list_error_x)
    avg_err_y, std_err_y = np.mean(list_error_y), np.std(list_error_y)
    
    avg_rmse, std_rmse = np.mean(list_rmse), np.std(list_rmse)
    
    print(f"Valid samples: {berhasil} of {len(IMAGE_SETS)} available\n")
    
    print("--- 1. TRAVEL & ERROR STATISTICS ---")
    print(f"Target               : X = {EXPECTED_DELTA_X:.4f} mm, Y = {EXPECTED_DELTA_Y:.4f} mm")
    print(f"Mean Delta X         : {avg_dx:8.4f} mm (Std Dev: {std_dx:.4f} mm)")
    print(f"Mean Delta Y         : {avg_dy:8.4f} mm (Std Dev: {std_dy:.4f} mm)")
    print(f"Mean Error X         : {avg_err_x:8.4f} mm (Std Dev: {std_err_x:.4f} mm)")
    print(f"Mean Error Y         : {avg_err_y:8.4f} mm (Std Dev: {std_err_y:.4f} mm)")
    print(f"Mean RMSE            : {avg_rmse:8.4f} px (Std Dev: {std_rmse:.4f} px)")
    
    print("\n--- 2. STEPS/MM UPDATE RECOMMENDATION ---")
    if EXPECTED_DELTA_Y != 0:
        new_steps_y = (EXPECTED_DELTA_Y / avg_dy) * CURRENT_STEPS_MM
        print(f"Current Steps/mm (Y) : {CURRENT_STEPS_MM:.4f}")
        print(f"-> NEW Steps/mm (Y)  : {new_steps_y:.4f}  (G-Code: M92 Y{new_steps_y:.2f})")
    else:
        print("-> Steps/mm Y update: skipped (target Delta Y = 0)")

    print("\n--- 3. BACKLASH COMPENSATION RECOMMENDATION ---")
    avg_backlash = np.mean(list_backlash)
    std_backlash = np.std(list_backlash)
    print(f"Mean Backlash Y      : {avg_backlash:8.4f} mm (Std Dev: {std_backlash:.4f} mm)")
    print(f"-> Firmware update : G-Code: M425 Y{avg_backlash:.4f}")

    print("\n--- 4. SKEW COMPENSATION RECOMMENDATION (XY ORTHOGONALITY) ---")
    avg_skew = np.mean(list_skew)
    angle_deg = np.degrees(np.arctan(avg_skew))
    print(f"Mean Skew (I)        : {avg_skew:.6f}")
    print(f"Axis tilt            : {angle_deg:.4f} Degrees")
    if abs(avg_skew) < 0.001:
        print("-> Skew status      : Orthogonal. No compensation needed.")
    else:
        print(f"-> Firmware update : G-Code: M852 I{avg_skew:.6f}")

else:
    print("No valid data passed the selection (RMSE > 0.2 or processing failed).")

# --- DISPLAY FAILED SAMPLES ---
print("\n--- 5. FAILED SAMPLE STATUS ---")
if len(list_sampel_gagal) > 0:
    print(f"There were {len(list_sampel_gagal)} rejected/failed samples before reaching {MAX_VALID_SAMPLES} valid samples:")
    print(f"-> {list_sampel_gagal}")
else:
    print(f"-> No samples failed before reaching the target of {MAX_VALID_SAMPLES} valid samples!")
print("="*70)

In [ ]:
# ==========================================================================
# [OPTIONAL / HISTORICAL] FIND NOISE
# ==========================================================================
# This cell was previously used to compare one reference image (calib_0000.png) with
# several other images from the "pengujian_noise_fix_banget" dataset to find the source
# of noise. That dataset is NOT included in this repo, so base_path below is deliberately
# pointed to a non-existent location: every pair will be reported as [FAILED] and
# no numbers will be produced. This is NOT an error.
# If you want to use it, point base_path to your own folder containing
# calib_0000.png, calib_0011.png, etc.
# ==========================================================================
#FIND NOISE
# =========================================================================================
import cv2
import numpy as np
import os

# --- 1. LOADING CAMERA PARAMETERS ---
try:
    K = camera_matrix 
    D = dist_coeffs
except NameError:
    raise NameError("ERROR: Run the Camera Calibration Cell first!")

CHESSBOARD_SIZE = (7, 7)
SQUARE_SIZE = 20.0 # mm
expected_pts = CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1]

objp = np.zeros((expected_pts, 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2) * SQUARE_SIZE

# --- 2. IMAGE INPUT CONFIGURATION ---
base_path = CFG.resolve("noise/pengujian_noise_fix_banget")   # not yet in repo

IMAGE_PAIRS = [
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0011.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0019.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0013.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0004.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0018.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0006.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0007.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0008.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0019.png")),
    (os.path.join(base_path, "calib_0000.png"), os.path.join(base_path, "calib_0010.png"))
]

# --- 3. POSE EXTRACTION FUNCTION (WITH DEDICATED SORTING IMAGE) ---
def get_pose_from_image(image_path, draw_debug=False):
    if not os.path.exists(image_path):
        return None, None, None
        
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Create the drawing canvas: grayscale but 3-channel so red and green can be drawn
    img_draw = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR) if draw_debug else None
    
    lines = detect_edlines_raw(gray)
    initial_corners, valid_lines = get_initial_corners_strict_paper(gray, lines)
    
    if len(initial_corners) >= expected_pts:
        refined_corners = optimize_subpixel_gradient(gray, initial_corners)
        sorted_corners = sort_corners_robust_paper(refined_corners, gray, CHESSBOARD_SIZE)
        
        if sorted_corners is not None:
            # --- DRAWING BLOCK: Exactly like your `img_f` ---
            if draw_debug:
                for idx, pt in enumerate(sorted_corners):
                    # Note: cv2.imshow uses BGR, so red is (0, 0, 255)
                    cv2.circle(img_draw, (int(pt[0]), int(pt[1])), 2, (0, 0, 255), -1)
                    # Using font type 1 (FONT_HERSHEY_PLAIN) exactly like your previous code
                    cv2.putText(img_draw, str(idx+1), (int(pt[0])+4, int(pt[1])-4), 1, 1.5, (0, 0, 255), 2)

            corners_final = sorted_corners.reshape(-1, 1, 2)
            success, rvec, tvec = cv2.solvePnP(objp, corners_final, K, D)
            
            if success:
                imgpts2, _ = cv2.projectPoints(objp, rvec, tvec, K, D)
                norm_l2 = cv2.norm(corners_final, imgpts2, cv2.NORM_L2)
                rmse = np.sqrt((norm_l2 ** 2) / len(imgpts2))
                return tvec, rmse, img_draw
                
    return None, None, img_draw

# --- 4. BATCH EXECUTION & VISUALIZATION ---
print("="*60)
print("POSITION & TRAVEL ANALYSIS")
print("="*60)

list_delta_x = []
list_delta_y = []

for idx, (origin_path, moved_path) in enumerate(IMAGE_PAIRS):
    print(f"\n[PAIR {idx+1}]")
    
    # Enable visualization ONLY for the first image pair
    is_visualize = (idx == 0) 
    
    tvec_origin, rmse_origin, img_orig_drawn = get_pose_from_image(origin_path, draw_debug=is_visualize)
    tvec_moved, rmse_moved, img_moved_drawn = get_pose_from_image(moved_path, draw_debug=is_visualize)
    
    if tvec_origin is not None and tvec_moved is not None:
        x_orig, y_orig = abs(tvec_origin[0][0]), abs(tvec_origin[1][0])
        x_moved, y_moved = abs(tvec_moved[0][0]), abs(tvec_moved[1][0])
        
        delta_x = x_moved - x_orig
        delta_y = y_moved - y_orig
        
        list_delta_x.append(delta_x)
        list_delta_y.append(delta_y)
        
        print(f"  > Image 1 (Initial) : |X| = {x_orig:8.4f} mm, |Y| = {y_orig:8.4f} mm | RMSE = {rmse_origin:.4f} px")
        print(f"  > Image 2 (Final)   : |X| = {x_moved:8.4f} mm, |Y| = {y_moved:8.4f} mm | RMSE = {rmse_moved:.4f} px")
        print(f"  > TRAVEL           : Delta X = {delta_x:8.4f} mm, Delta Y = {delta_y:8.4f} mm")

        # --- FRAME TEXT & COORDINATE ADDITION BLOCK ---
        if is_visualize and img_orig_drawn is not None and img_moved_drawn is not None:
            font = cv2.FONT_HERSHEY_SIMPLEX
            
            # Initial Frame text
            cv2.putText(img_orig_drawn, "INITIAL FRAME", (30, 50), font, 1.2, (0, 0, 255), 3) 
            cv2.putText(img_orig_drawn, f"X: {x_orig:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2) 
            cv2.putText(img_orig_drawn, f"Y: {y_orig:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2) 
            
            # Final Frame text
            cv2.putText(img_moved_drawn, "FINAL FRAME", (30, 50), font, 1.2, (0, 0, 255), 3)
            cv2.putText(img_moved_drawn, f"X: {x_moved:.3f} mm", (30, 90), font, 0.9, (0, 255, 0), 2)
            cv2.putText(img_moved_drawn, f"Y: {y_moved:.3f} mm", (30, 130), font, 0.9, (0, 255, 0), 2)
            
            # Save the images as separate PNGs
            cv2.imwrite("visualisasi_awal.png", img_orig_drawn)
            cv2.imwrite("visualisasi_akhir.png", img_moved_drawn)
            print("\n  [INFO] Images 'visualisasi_awal.png' and 'visualisasi_akhir.png' saved successfully!")
            
    else:
        print("  [FAILED] Could not extract corners from one or both images.")

# --- 5. STATISTICAL SUMMARY ---
print("\n" + "="*60)
print("FINAL AVERAGE ANALYSIS RESULTS (FROM ABSOLUTE COORDINATES)")
print("="*60)

if len(list_delta_x) > 0:
    avg_dx = np.mean(list_delta_x)
    avg_dy = np.mean(list_delta_y)
    std_dx = np.std(list_delta_x)
    std_dy = np.std(list_delta_y)
    
    print(f"Successful pairs         : {len(list_delta_x)} / {len(IMAGE_PAIRS)}")
    print(f"Mean Delta X             : {avg_dx:8.4f} mm (Std Dev: {std_dx:.4f})")
    print(f"Mean Delta Y             : {avg_dy:8.4f} mm (Std Dev: {std_dy:.4f})")
else:
    print("No data could be processed to compute the averages.")

print("="*60)
print("DONE")

In [ ]:
import glob
import os
import cv2


def cek_frame_folder(folder_path):
    # Ensure the folder exists
    if not os.path.exists(folder_path):
        print(f"[!] Folder not found: {folder_path}")
        return

    # Find all common video files (.avi, .mp4, .mov)
    extensions = ["*.avi", "*.mp4", "*.mov"]
    video_files = []
    for ext in extensions:
        video_files.extend(glob.glob(os.path.join(folder_path, ext)))

    # Sort file names alphabetically
    video_files.sort()

    if not video_files:
        print(f"No video files found in: {folder_path}")
        return

    print(f"\nScanning folder: {folder_path}")
    print("=" * 65)
    print(f"{'File Name':<30} | {'Resolution':<15} | {'FPS':<6} | {'Total Frames'}")
    print("=" * 65)

    total_semua_frame = 0

    for vid_path in video_files:
        nama_file = os.path.basename(vid_path)
        cap = cv2.VideoCapture(vid_path)

        if not cap.isOpened():
            print(f"{nama_file:<30} | [!] Failed to open video")
            continue

        # Get video metadata
        lebar = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        tinggi = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        total_semua_frame += total_frame

        # Truncate long file names to keep the table tidy
        nama_file_disp = (
            (nama_file[:27] + "...") if len(nama_file) > 30 else nama_file
        )

        print(
            f"{nama_file_disp:<30} | {lebar}x{tinggi:<10} | {fps:<6.0f} | {total_frame:,} frame"
        )

        cap.release()

    print("=" * 65)
    print(f"Total Frames Overall: {total_semua_frame:,} frames")
    print(f"Total Videos Processed : {len(video_files)} video\n")


if __name__ == "__main__":
    # --- HOW TO USE ---
    # Change the path below to match your video folder name.
    # Example: "dataset_otomatis_sesi_20260512_143000/video"
    TARGET_FOLDER = CFG.VIDEO_DIR  # set via config_paths.py

    cek_frame_folder(TARGET_FOLDER)